# IAC Scientific Publication ↔ Press Note Matching

A compact, reproducible demonstration of the publication-to-press-note ranking workflow. The notebook uses the same package as the CLI and runs **offline by default**.

> The 0–100 value is a heuristic ranking score, not a calibrated probability. Candidate links remain subject to manual review.

## 1. Imports and configuration

The public notebook uses bundled fixtures rather than live scraping so that results remain reproducible in GitHub, Jupyter and hosted notebook environments.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from iac_matching.core import extract_articles_from_txt, match_articles_to_press_notes
from iac_matching.evaluation import evaluate_ranking

THRESHOLD = 72.0
TOP_K = 8
print({"threshold": THRESHOLD, "top_k": TOP_K, "mode": "offline fixture"})

{'threshold': 72.0, 'top_k': 8, 'mode': 'offline fixture'}


## 2. Public sample

The public repository contains **29 numbered publication titles** and a compact press-note fixture. The earlier local workflow used a larger 752-title source list, which is intentionally not published here.

In [2]:
articles = extract_articles_from_txt((ROOT / "data/sample_articles.txt").read_text(encoding="utf-8"))
press_notes = pd.read_csv(ROOT / "data/sample_press_notes.csv", dtype={"press_note_id": str})
validation = pd.read_csv(ROOT / "data/validation_pairs.csv", dtype={"article_id": str})

print(f"Publication titles: {len(articles)}")
print(f"Press-note fixtures: {len(press_notes)}")
print(f"Reference pairs: {len(validation)}")
for article in articles[:3]:
    print(f"  {article.article_id}: {article.title}")

Publication titles: 29
Press-note fixtures: 8
Reference pairs: 9
  77: Constraining stellar and orbital co-evolution through ensemble seismology of solar-like oscillators in binary systems. A census of oscillating red giants and dwarf stars in Gaia DR3 binaries
  80: Cosmology and fundamental physics with the ELT-ANDES spectrograph
  87: Feeding and feedback processes in the Spiderweb proto-intracluster medium


## 3. Rank candidate links

The original interpretable strategy is preserved: RapidFuzz token-set similarity, partial similarity, exact-title evidence and shared scientific terms are combined into a review score.

In [3]:
matches = match_articles_to_press_notes(press_notes, articles, threshold=THRESHOLD, top_k=TOP_K)
output_path = ROOT / "outputs/iac_article_pressnote_matches.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
matches.to_csv(output_path, index=False, encoding="utf-8")

review = matches[["press_note_title", "article_id", "article_title", "score", "match_tier"]]
print(review.to_string(index=False, max_colwidth=72))
print(f"\nExported {len(matches)} candidate rows to {output_path}")

                                                        press_note_title article_id                                                            article_title  score  match_tier
Científicos del IAC detectan una ‘sub-tierra’ que orbita Barnard, la ...        549                          A sub-Earth-mass planet orbiting Barnard's star 100.00 very_strong
Descubren un exoplaneta del tamaño de la Tierra orbitando una estrell...        380 Detection of an Earth-sized exoplanet orbiting the nearby ultracool d... 100.00 very_strong
  Un exoplaneta excéntrico revela cómo se forman los jupíteres calientes        430           A hot-Jupiter progenitor on a super-eccentric retrograde orbit 100.00 very_strong
Hallan evidencias de que las supergigantes azules podrían nacer de la...        161 Evidence for Evolved Stellar Binary Mergers in Observed B-type Blue S... 100.00 very_strong
La inteligencia artificial revela la historia de la fusión de galaxia...        556 Constraints on the in situ and ex si

## 4. Regression check

The bundled reference mapping is a **software-regression fixture**, not a generalisation benchmark. It verifies that known sample links remain highly ranked after refactors.

In [4]:
metrics = evaluate_ranking(press_notes, articles, validation)
metrics_path = ROOT / "outputs/regression_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2) + "\n", encoding="utf-8")
print(json.dumps(metrics, indent=2))

{
  "evaluated_press_notes": 8,
  "relevant_pairs": 9,
  "mrr": 1.0,
  "pair_recall_at_1": 0.8889,
  "note_hit_rate_at_1": 1.0,
  "pair_recall_at_3": 1.0,
  "note_hit_rate_at_3": 1.0,
  "pair_recall_at_5": 1.0,
  "note_hit_rate_at_5": 1.0
}


## 5. Interpretation and limitations

- Candidate links require manual verification.
- The ranking score is heuristic and not probabilistically calibrated.
- The public fixture is intentionally small and cannot estimate archive-wide precision/recall.
- English paper titles and Spanish outreach text make indirect semantic matches difficult.
- Live scraping is available through the CLI but is deliberately excluded from the default notebook and CI path.

See [`docs/METHODOLOGY.md`](docs/METHODOLOGY.md) for additional methodology notes.